# Fraud Scoring — Auto Insurance Claims

**Dataset:** Kaggle — Auto Insurance Claims (buntyshah)  
**Target:** `fraud_reported` (Y/N)  
**Model:** XGBoost con SHAP para explicabilidad  
**Deploy:** SageMaker Serverless Inference (costo ~$0 en volumen bajo)

## Pasos de seguridad aplicados (antes de entrenar)

| Paso | Tipo | Columnas afectadas |
|------|------|--------------------|
| 1 | Supresión de identificadores directos | `policy_number`, `incident_location`, `policy_bind_date` |
| 2 | Supresión de quasi-identificadores | `insured_zip`, `insured_city` |
| 3 | Generalización de fecha y edad | `incident_date` → mes, `age` → bucket de 10 años |
| 4 | Privacidad diferencial (ruido Laplace) | `total_claim_amount`, `capital_gains`, `capital_loss` |
| 5 | Verificación k-anonimidad (k≥5) | Combinación de quasi-identificadores residuales |

In [ ]:
# Instalar dependencias (ejecutar solo si no están instaladas)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import json
import os
import tarfile
import boto3
import sagemaker

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    precision_recall_curve, average_precision_score
)
from sklearn.utils.class_weight import compute_sample_weight

np.random.seed(42)
print('Libraries loaded OK')

## 1. Descarga del dataset

**Opción A — Kaggle API (recomendado):**
```bash
# Configurar credenciales primero:
# ~/.kaggle/kaggle.json  →  {"username":"...","key":"..."}
kaggle datasets download -d buntyshah/auto-insurance-claims-data
unzip auto-insurance-claims-data.zip
```

**Opción B — Manual:** descargar `insurance_claims.csv` desde Kaggle y subirlo aquí.

In [ ]:
# Descomenta si usas Kaggle API
# import subprocess
# subprocess.run(['kaggle', 'datasets', 'download', '-d', 'buntyshah/auto-insurance-claims-data'], check=True)
# subprocess.run(['unzip', '-o', 'auto-insurance-claims-data.zip'], check=True)

DATASET_PATH = 'insurance_claims.csv'
assert os.path.exists(DATASET_PATH), f'Dataset no encontrado: {DATASET_PATH}'

raw = pd.read_csv(DATASET_PATH)
print(f'Shape: {raw.shape}')
print(f'Columnas: {list(raw.columns)}')
raw.head(3)

## 2. Security Preprocessing — Pasos 1-3: Supresión y Generalización

In [ ]:
df = raw.copy()

# ── Paso 1: Supresión de identificadores directos ────────────────────────────
DROP_PII = ['policy_number', 'incident_location', 'policy_bind_date']
# '_c39' es la columna sin nombre que aparece al final del CSV de Kaggle
DROP_PII += [c for c in df.columns if c.startswith('_c') or c.startswith('Unnamed')]
df.drop(columns=[c for c in DROP_PII if c in df.columns], inplace=True)
print(f'Paso 1 — Eliminadas {len(DROP_PII)} columnas PII directas')

# ── Paso 2: Supresión de quasi-identificadores ───────────────────────────────
DROP_QUASI = ['insured_zip', 'insured_city']
df.drop(columns=[c for c in DROP_QUASI if c in df.columns], inplace=True)
# incident_state se queda: es demasiado grueso para re-identificar
print(f'Paso 2 — Eliminadas {len(DROP_QUASI)} columnas quasi-identificadoras')

# ── Paso 3: Generalización de fecha y edad ───────────────────────────────────
# Fecha → solo mes (elimina el día que permite triangulación)
df['incident_month'] = pd.to_datetime(df['incident_date']).dt.month
df.drop(columns=['incident_date'], inplace=True)

# Edad → bucket de 10 años (30-39, 40-49, ...)
df['age_bucket'] = (df['age'] // 10) * 10
df.drop(columns=['age'], inplace=True)

print(f'Paso 3 — incident_date → incident_month, age → age_bucket')
print(f'Shape después de supresión/generalización: {df.shape}')

## 3. Security Preprocessing — Paso 4: Privacidad Diferencial

In [ ]:
# Mecanismo de Laplace: agrega ruido calibrado para proteger valores exactos.
# Sensibilidad = rango del atributo. ε=1.0 es el presupuesto de privacidad estándar.
EPSILON = 1.0

def laplace_noise(series: pd.Series, epsilon: float) -> pd.Series:
    sensitivity = series.max() - series.min()
    if sensitivity == 0:
        return series
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale, len(series))
    return (series + noise).clip(lower=0)

FINANCIAL_COLS = ['total_claim_amount', 'injury_claim', 'property_claim',
                  'vehicle_claim', 'capital_gains', 'capital_loss']

for col in [c for c in FINANCIAL_COLS if c in df.columns]:
    original_mean = df[col].mean()
    df[col] = laplace_noise(df[col], EPSILON)
    print(f'{col}: media {original_mean:.0f} → {df[col].mean():.0f} (ruido aplicado)')

print(f'\nPaso 4 — Ruido Laplace aplicado a {len(FINANCIAL_COLS)} columnas financieras')

## 4. Security Preprocessing — Paso 5: Verificación k-Anonimidad

In [ ]:
# Verifica que ninguna combinación de quasi-identificadores residuales
# identifica a menos de k=5 individuos (protección contra linkage attacks).
K_MIN = 5

quasi_identifiers = ['age_bucket', 'insured_sex', 'incident_state', 'incident_month']
quasi_identifiers = [c for c in quasi_identifiers if c in df.columns]

group_sizes = df.groupby(quasi_identifiers).size()
min_group = group_sizes.min()
violating = (group_sizes < K_MIN).sum()

print(f'Quasi-identificadores usados: {quasi_identifiers}')
print(f'Grupo mínimo: {min_group} registros')
print(f'Grupos con < {K_MIN} registros: {violating}')

if violating > 0:
    print(f'⚠ {violating} grupos violan k={K_MIN}. Opciones:')
    print('  - Suprimir esas filas')
    print('  - Generalizar más (age_bucket → quinquenios de 20 años)')
    # Supresión conservadora: eliminar grupos pequeños
    valid_groups = group_sizes[group_sizes >= K_MIN].index
    df = df.set_index(quasi_identifiers).loc[valid_groups].reset_index()
    print(f'  → Supresión aplicada. Shape: {df.shape}')
else:
    print(f'✓ k-anonimidad k={K_MIN} verificada')

print(f'\nShape final del dataset seguro: {df.shape}')

## 5. EDA — Exploración del Dataset Seguro

In [ ]:
# Target distribution
df['fraud_reported'] = df['fraud_reported'].map({'Y': 1, 'N': 0})
fraud_rate = df['fraud_reported'].mean()
print(f'Tasa de fraude: {fraud_rate:.1%} ({df["fraud_reported"].sum()} / {len(df)})')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Target distribution
df['fraud_reported'].value_counts().plot(kind='bar', ax=axes[0], color=['#2196F3', '#F44336'])
axes[0].set_title('Distribución: Fraude vs No Fraude')
axes[0].set_xticklabels(['No Fraude', 'Fraude'], rotation=0)
axes[0].set_ylabel('Registros')

# 2. Claim amount by fraud
df.groupby('fraud_reported')['total_claim_amount'].plot(kind='hist', ax=axes[1], bins=30, alpha=0.7)
axes[1].set_title('Monto reclamado por clase')
axes[1].legend(['No Fraude', 'Fraude'])

# 3. Incident severity by fraud
sev_fraud = df.groupby(['incident_severity', 'fraud_reported']).size().unstack(fill_value=0)
sev_fraud.plot(kind='bar', ax=axes[2], color=['#2196F3', '#F44336'])
axes[2].set_title('Severidad por clase')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45)
axes[2].legend(['No Fraude', 'Fraude'])

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Feature Engineering

In [ ]:
df_model = df.copy()

# Columnas categóricas a encodear
CATEGORICAL = [
    'insured_sex', 'insured_education_level', 'insured_occupation',
    'insured_hobbies', 'insured_relationship',
    'incident_type', 'collision_type', 'incident_severity',
    'authorities_contacted', 'incident_state',
    'property_damage', 'police_report_available',
    'auto_make', 'auto_model',
]
CATEGORICAL = [c for c in CATEGORICAL if c in df_model.columns]

le = LabelEncoder()
label_encoders = {}
for col in CATEGORICAL:
    df_model[col] = df_model[col].fillna('UNKNOWN')
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    label_encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))

# Rellenar nulos numéricos con mediana
numeric_cols = df_model.select_dtypes(include=[np.number]).columns
df_model[numeric_cols] = df_model[numeric_cols].fillna(df_model[numeric_cols].median())

# Guardar encoding para usarlo en Lambda
with open('label_encoders.json', 'w') as f:
    json.dump(label_encoders, f, indent=2)

print(f'Columnas categóricas encodeadas: {len(CATEGORICAL)}')
print(f'Nulls restantes: {df_model.isnull().sum().sum()}')
print(f'Shape final: {df_model.shape}')

In [ ]:
# Definir features y target
TARGET = 'fraud_reported'

# Features usadas en entrenamiento — DEBEN coincidir con el orden en Lambda
# Ver sección 10 para el mapeo exacto SF → features
FEATURE_COLS = [
    'total_claim_amount',
    'injury_claim',
    'property_claim',
    'vehicle_claim',
    'number_of_vehicles_involved',
    'bodily_injuries',
    'witnesses',
    'property_damage',          # binary: YES=1, NO=0
    'police_report_available',  # binary: YES=1, NO=0
    'incident_severity',        # ordinal encoded
    'incident_type',
    'collision_type',
    'authorities_contacted',
    'incident_month',
    'age_bucket',
    'months_as_customer',
    'insured_sex',
    'insured_education_level',
    'insured_occupation',
    'insured_hobbies',
    'insured_relationship',
    'incident_state',
    'auto_make',
    'auto_model',
    'auto_year',
    'capital_gains',
    'capital_loss',
    'policy_deductable',
    'policy_annual_premium',
    'umbrella_limit',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_model.columns]

X = df_model[FEATURE_COLS]
y = df_model[TARGET]

print(f'Features: {len(FEATURE_COLS)}')
print(f'Target: {TARGET} — fraude={y.sum()} / total={len(y)}')

# Guardar lista de features para usar en Lambda
with open('feature_columns.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
print('feature_columns.json guardado')

## 7. Entrenamiento XGBoost

In [ ]:
# Split estratificado para mantener proporción de fraude en todos los conjuntos
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.18, random_state=42, stratify=y_temp
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
print(f'Fraude en train: {y_train.mean():.1%}')

# scale_pos_weight compensa el desbalance de clases sin oversampling
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

In [ ]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

print(f'\nMejor iteración: {model.best_iteration}')
print(f'Mejor AUC (val): {model.best_score:.4f}')

## 8. Evaluación

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)
ap  = average_precision_score(y_test, y_pred_proba)

print(f'AUC-ROC: {auc:.4f}')
print(f'Average Precision: {ap:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Fraude', 'Fraude']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Fraude', 'Fraude'],
            yticklabels=['No Fraude', 'Fraude'])
axes[0].set_title(f'Confusion Matrix (AUC={auc:.3f})')

# Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
axes[1].plot(recall, precision, 'b-', linewidth=2)
axes[1].axhline(y=fraud_rate, color='r', linestyle='--', label=f'Baseline ({fraud_rate:.1%})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall (AP={ap:.3f})')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. SHAP — Explicabilidad

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary plot — importancia global
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP — Importancia Global de Features')
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()

# Top 10 features por importancia media
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': np.abs(shap_values).mean(axis=0)
}).sort_values('importance', ascending=False)

print('Top 10 features más relevantes:')
print(feature_importance.head(10).to_string(index=False))

## 10. Guardar Modelo — Formato SageMaker XGBoost

In [ ]:
import os, tarfile

os.makedirs('model', exist_ok=True)

# SageMaker XGBoost built-in espera el archivo en formato binario XGBoost (xgb.Booster)
# El nombre del archivo DEBE ser 'xgboost-model'
model.save_model('model/xgboost-model')

# Empaquetar en model.tar.gz (formato requerido por SageMaker)
with tarfile.open('model.tar.gz', 'w:gz') as tar:
    tar.add('model/xgboost-model', arcname='xgboost-model')

print('Modelo guardado: model/xgboost-model')
print('Comprimido en:   model.tar.gz')

# Guardar también metadata del modelo
metadata = {
    'feature_columns': FEATURE_COLS,
    'label_encoders': label_encoders,
    'scale_pos_weight': float(scale_pos_weight),
    'best_iteration': int(model.best_iteration),
    'auc_val': float(model.best_score),
    'auc_test': float(auc),
    'average_precision_test': float(ap),
    'fraud_rate_train': float(y_train.mean()),
    'feature_importances': feature_importance.to_dict(orient='records'),
}
with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('Metadata guardada: model_metadata.json')

## 11. Deploy — SageMaker Serverless Inference

**Costo esperado:**
- Sin tráfico: $0
- 10,000 invocaciones/mes (~100ms/invocación): < $0.01
- Cold start: 300-800ms (irrelevante dentro de Step Function)

In [ ]:
import boto3
import sagemaker
from sagemaker import get_execution_role
from datetime import datetime

# Configuración
sess        = sagemaker.Session()
region      = sess.boto_region_name
bucket      = sess.default_bucket()
role        = get_execution_role()
model_name  = f'fraud-scoring-xgboost-{datetime.now().strftime("%Y%m%d-%H%M")}'
endpoint_name = 'fraud-scoring-serverless'

print(f'Region: {region}')
print(f'Bucket: {bucket}')
print(f'Model:  {model_name}')
print(f'Endpoint: {endpoint_name}')

In [ ]:
# Subir model.tar.gz a S3
s3_model_path = sess.upload_data(
    path='model.tar.gz',
    bucket=bucket,
    key_prefix='fraud-scoring/model',
)
print(f'Modelo en S3: {s3_model_path}')

In [ ]:
# Imagen XGBoost built-in de SageMaker (no requiere código propio)
from sagemaker.image_uris import retrieve

xgboost_image = retrieve(
    framework='xgboost',
    region=region,
    version='1.7-1',
    image_scope='inference',
)
print(f'Imagen: {xgboost_image}')

sm_client = boto3.client('sagemaker', region_name=region)

# Crear SageMaker Model
sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={
        'Image': xgboost_image,
        'ModelDataUrl': s3_model_path,
        'Environment': {
            'SAGEMAKER_CONTAINER_LOG_LEVEL': '20',
        },
    },
    ExecutionRoleArn=role,
)
print(f'Modelo SageMaker creado: {model_name}')

In [ ]:
# Crear Endpoint Config con Serverless Inference
# memory_size_in_mb: 1024 es suficiente para XGBoost con 30 features
# max_concurrency: 5 invocaciones simultáneas
config_name = f'{endpoint_name}-config'

sm_client.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': model_name,
            'ServerlessConfig': {
                'MemorySizeInMB': 1024,
                'MaxConcurrency': 5,
            },
        }
    ],
)
print(f'Endpoint config creado: {config_name}')

In [ ]:
import time

# Crear o actualizar endpoint
try:
    sm_client.create_endpoint(
        EndpointName=endpoint_name,
        EndpointConfigName=config_name,
    )
    print(f'Endpoint creado: {endpoint_name}')
except sm_client.exceptions.ClientError as e:
    if 'Cannot create already existing endpoint' in str(e):
        sm_client.update_endpoint(
            EndpointName=endpoint_name,
            EndpointConfigName=config_name,
        )
        print(f'Endpoint actualizado: {endpoint_name}')
    else:
        raise

# Esperar a que esté listo
print('Esperando endpoint...', end='')
while True:
    status = sm_client.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus']
    if status == 'InService':
        print(f' ✓ {status}')
        break
    elif status in ('Failed', 'RollingBack'):
        raise RuntimeError(f'Endpoint falló: {status}')
    print('.', end='', flush=True)
    time.sleep(15)

print(f'\n🎉 Endpoint listo: {endpoint_name}')

## 12. Test de Inferencia

In [ ]:
import json

smr_client = boto3.client('sagemaker-runtime', region_name=region)

# Tomar un ejemplo del test set
sample = X_test.iloc[0:3].values.tolist()
payload = '\n'.join([','.join([str(v) for v in row]) for row in sample])

# XGBoost built-in acepta CSV por defecto
response = smr_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='text/csv',
    Accept='text/csv',
    Body=payload,
)

result = response['Body'].read().decode('utf-8').strip().split('\n')
probabilities = [float(r) for r in result]

for i, (prob, actual) in enumerate(zip(probabilities, y_test.iloc[0:3])):
    score = round(prob * 100)
    label = 'FRAUDE' if actual == 1 else 'OK'
    print(f'Muestra {i+1}: score={score}/100 | prob_fraude={prob:.3f} | real={label}')

## 13. Integración con Lambda (aggregate-risk)

Después del deploy, configurar el endpoint en la Lambda:

```bash
# Obtener nombre de la función
FUNCTION_NAME=$(aws cloudformation describe-stacks \
  --stack-name AssistanceStack-Prod \
  --query "Stacks[0].Outputs[?ExportName=='assistance-prod-aggregate-risk-function-name'].OutputValue" \
  --output text 2>/dev/null || echo "assistance-prod-aggregate-risk")

# Actualizar variable de entorno
aws lambda update-function-configuration \
  --function-name $FUNCTION_NAME \
  --environment "Variables={FRAUD_SCORING_ENDPOINT_NAME=fraud-scoring-serverless}"
```

### Mapeo de features SF → modelo

| Feature del modelo | Fuente en Step Function |
|--------------------|-------------------------|
| `total_claim_amount` | `estimatedAmount` |
| `property_damage` | `possibleAlteration` (bool → 0/1) |
| `police_report_available` | `!lowQualityDocument` (bool → 0/1) |
| `number_of_vehicles_involved` | `involvedParties.length` |
| `incident_severity` | `integrityScore` → bucket 1-4 |
| `bodily_injuries` | default 0 |
| `witnesses` | default 0 |
| otras categorícas | mediana/moda del training set |

In [ ]:
# Calcular valores por defecto (mediana/moda) para features desconocidas en Lambda
defaults = {}
for col in FEATURE_COLS:
    if df_model[col].dtype in [np.float64, np.int64]:
        defaults[col] = float(df_model[col].median())
    else:
        defaults[col] = float(df_model[col].mode()[0])

with open('feature_defaults.json', 'w') as f:
    json.dump({'feature_columns': FEATURE_COLS, 'defaults': defaults}, f, indent=2)

print('feature_defaults.json guardado — usar en Lambda para features desconocidas')
print(f'\nEjemplo de defaults:')
for col in ['total_claim_amount', 'months_as_customer', 'incident_severity']:
    if col in defaults:
        print(f'  {col}: {defaults[col]}')

In [ ]:
# Resumen final
print('═' * 55)
print('  RESUMEN DEL MODELO')
print('═' * 55)
print(f'  AUC-ROC (test):         {auc:.4f}')
print(f'  Average Precision:      {ap:.4f}')
print(f'  Features:               {len(FEATURE_COLS)}')
print(f'  Tasa de fraude (train): {y_train.mean():.1%}')
print(f'  Endpoint:               {endpoint_name}')
print(f'  Tipo:                   Serverless Inference')
print(f'  Costo estimado:         ~$0 en volumen de práctica')
print('═' * 55)
print()
print('Archivos generados:')
print('  model/xgboost-model    → modelo entrenado')
print('  model.tar.gz           → empaquetado para SageMaker')
print('  feature_columns.json   → orden exacto de features')
print('  feature_defaults.json  → valores por defecto para Lambda')
print('  label_encoders.json    → encoding de categorícas')
print('  model_metadata.json    → métricas y metadata')
print('  eda_overview.png       → distribuciones')
print('  model_evaluation.png   → métricas visuales')
print('  shap_summary.png       → importancia de features')